Image Encoder (frozen)

Q-Former with learnable query tokens + cross-attention

Text Encoder (for ITC + ITM)

LLM-like Decoder

ITG loss, ITC loss, ITM loss

Full train step

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model


# ============================================================
# IMAGE ENCODER (Frozen)
# ============================================================

def build_image_encoder(embed_dim=256):
    inputs = layers.Input(shape=(128, 128, 3))
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    # flatten spatial dims → patch sequence
    x = layers.Reshape((-1, 128))(x)
    x = layers.Dense(embed_dim)(x)

    model = Model(inputs, x, name="image_encoder")
    model.trainable = False
    return model


# ============================================================
# BASIC TRANSFORMER BLOCKS for Q-Former
# ============================================================

class TransformerBlock(layers.Layer):
    def __init__(self, hidden_dim, num_heads=4, mlp_ratio=4, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(num_heads=num_heads,
                                             key_dim=hidden_dim)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = tf.keras.Sequential([
            layers.Dense(hidden_dim * mlp_ratio, activation="relu"),
            layers.Dense(hidden_dim),
        ])
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x):
        attn_out = self.mha(x, x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class CrossAttention(layers.Layer):
    def __init__(self, hidden_dim, num_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(num_heads=num_heads,
                                             key_dim=hidden_dim)
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, query, key_value):
        attn_out = self.mha(query=query, value=key_value, key=key_value)
        return self.norm(query + attn_out)


# ============================================================
# Q-FORMER (Query Tokens + Cross-Attn + Self-Attn)
# ============================================================

class QFormer(Model):
    def __init__(self, num_query_tokens=32, hidden_dim=256,
                 num_heads=4, num_layers=2):
        super().__init__()
        self.num_query_tokens = num_query_tokens
        self.hidden_dim = hidden_dim

        # Learnable query token matrix 32×256
        self.query_tokens = self.add_weight(
            shape=(num_query_tokens, hidden_dim),
            initializer="random_normal",
            trainable=True,
            name="query_tokens"
        )

        self.cross_attn = CrossAttention(hidden_dim, num_heads)
        self.blocks = [TransformerBlock(hidden_dim, num_heads)
                       for _ in range(num_layers)]

    def call(self, img_features):
        B = tf.shape(img_features)[0]

        # duplicate query tokens for the batch
        q = tf.tile(tf.expand_dims(self.query_tokens, 0), [B, 1, 1])

        # queries attend to image features
        q = self.cross_attn(q, img_features)

        # Q-Former transformer layers
        for blk in self.blocks:
            q = blk(q)

        return q  # (B, 32, D)


# ============================================================
# Q-Former's TEXT ENCODER (for ITC + ITM) to encode the text in the input (image, text) pair
# ============================================================

class TinyTextEncoder(Model):
    def __init__(self, vocab_size, hidden_dim=256, num_heads=4, num_layers=2):
        super().__init__()
        self.embed = layers.Embedding(vocab_size, hidden_dim)
        self.blocks = [TransformerBlock(hidden_dim, num_heads)
                       for _ in range(num_layers)]
        self.pooler = layers.Dense(hidden_dim)

    def call(self, input_ids):
        x = self.embed(input_ids)   # (B, T, D)
        for blk in self.blocks:
            x = blk(x)
        x = tf.reduce_mean(x, axis=1)  # mean pool → (B, D)
        return self.pooler(x)


# ============================================================
# DECODER (LLM-like)
# ============================================================

class TinyDecoder(Model):
    def __init__(self, vocab_size, hidden_dim=256,
                 num_heads=4, num_layers=2, max_len=32):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.max_len = max_len

        self.token_embed = layers.Embedding(vocab_size, hidden_dim)
        self.pos_embed = self.add_weight(
            shape=(max_len, hidden_dim),
            initializer="zeros",
            trainable=True
        )

        self.blocks = [TransformerBlock(hidden_dim, num_heads)
                       for _ in range(num_layers)]

        self.cross_attn = CrossAttention(hidden_dim, num_heads)
        self.lm_head = layers.Dense(vocab_size)

    def call(self, input_ids, visual_tokens):
        x = self.token_embed(input_ids)
        T = tf.shape(input_ids)[1]
        x = x + self.pos_embed[None, :T, :]

        # self-attention layers
        for blk in self.blocks:
            x = blk(x)

        # cross-attend to Q-Former visual tokens
        x = self.cross_attn(x, visual_tokens)

        return self.lm_head(x)


# ============================================================
# BLIP-2 MODEL WRAPPER
# ============================================================

class MiniBLIP2(Model):
    def __init__(self, vocab_size=1000):
        super().__init__()
        ## image encoder
        self.encoder = build_image_encoder(embed_dim=256)

        ## Q-former () & text encoder for Q-Former
        self.qformer = QFormer(hidden_dim=256)
        self.txt_encoder = TinyTextEncoder(vocab_size=vocab_size,
                                           hidden_dim=256)
        ## as LLM
        self.decoder = TinyDecoder(vocab_size=vocab_size, hidden_dim=256)

    def call(self, images, decoder_input_ids, encoder_input_ids):
        img_feats = self.encoder(images)
        visual_tokens = self.qformer(img_feats)
        text_embeds = self.txt_encoder(encoder_input_ids)
        logits = self.decoder(decoder_input_ids, visual_tokens)
        return logits, visual_tokens, text_embeds


# ============================================================
# BLIP-2 LOSSES (ITG + ITC + ITM)
# ============================================================

# --- ITG (generation loss) ---
loss_itg_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# --- ITC (contrastive loss) ---
def itc_loss(image_embeds, text_embeds, temperature=0.07):
    img = tf.math.l2_normalize(image_embeds, axis=-1)
    txt = tf.math.l2_normalize(text_embeds, axis=-1)

    logits = tf.matmul(img, txt, transpose_b=True) / temperature
    labels = tf.range(tf.shape(img)[0])

    ce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    return (ce(labels, logits) + ce(labels, tf.transpose(logits))) / 2.0

# --- ITM (matching loss) ---
class ITMHead(Model):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.net = tf.keras.Sequential([
            layers.Dense(hidden_dim, activation="relu"),
            layers.Dense(1)
        ])

    def call(self, img_emb, txt_emb):
        return self.net(tf.concat([img_emb, txt_emb], axis=-1))

itm_head = ITMHead()
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def itm_loss_function(img_embeds, text_embeds):
    B = tf.shape(img_embeds)[0]

    pos_logits = itm_head(img_embeds, text_embeds)
    pos_labels = tf.ones((B, 1))

    # shuffled text for negatives
    idx = tf.random.shuffle(tf.range(B))
    neg_logits = itm_head(img_embeds, tf.gather(text_embeds, idx))
    neg_labels = tf.zeros((B, 1))

    loss_pos = bce(pos_labels, pos_logits)
    loss_neg = bce(neg_labels, neg_logits)
    return (loss_pos + loss_neg) / 2.0


# ============================================================
# TRAIN STEP (FULL BLIP-2 LOSS)
# ============================================================

model = MiniBLIP2(vocab_size=1000)
optimizer = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(images, decoder_input_ids, decoder_target_ids, encoder_input_ids):
    with tf.GradientTape() as tape:
        logits, visual_tokens, text_embeds = model(
            images, decoder_input_ids, encoder_input_ids
        )

        img_embeds = tf.reduce_mean(visual_tokens, axis=1)

        loss_itg = loss_itg_fn(decoder_target_ids, logits)
        loss_itc = itc_loss(img_embeds, text_embeds)
        loss_itm = itm_loss_function(img_embeds, text_embeds)

        loss = loss_itg + 0.5 * loss_itc + 0.5 * loss_itm

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss, loss_itg, loss_itc, loss_itm


# ============================================================
# DUMMY TEST RUN
# ============================================================

B, T = 2, 6   # simulate training on 2 images, each with a text length of 6 tokens
dummy_images = tf.random.normal((B, 128, 128, 3))

# simulates decoder input tokens, vocabulary size simulated as 1000 tokens
# simulate correspondance to the "prefix" tokens fed into the LLM (T5, OPT, etc.)
decoder_in = tf.random.uniform((B, T), maxval=1000, dtype=tf.int32)

# shape: (B, T) = (2, 6). Contains the text you want the LLM to output given the image
# Used in the ITG (Image-Text Generation Loss) through cross-entropy
# i.e. LLM(predictions) vs decoder_tgt
decoder_tgt = tf.random.uniform((B, T), maxval=1000, dtype=tf.int32)

# simulates the input text tokens for the Q-Former encoder
# used for Contrastive loss (ITC) to compare image embedding vs text embedding
# used for Image–Text Matching (ITM) to classify whether the image and text match
encoder_in = tf.random.uniform((B, T), maxval=1000, dtype=tf.int32)

loss, l_itg, l_itc, l_itm = train_step(
    dummy_images, decoder_in, decoder_tgt, encoder_in
)

print("Total loss:", float(loss))
print("ITG loss:", float(l_itg))
print("ITC loss:", float(l_itc))
print("ITM loss:", float(l_itm))


Total loss: 7.707178592681885
ITG loss: 6.9774250984191895
ITC loss: 0.7536031007766724
ITM loss: 0.7059036493301392


In [2]:
# ============================================================
# DUMMY IMAGE + TEXT PAIR TEST
# ============================================================

# Let's build a single dummy batch:
B = 1           # batch size
T = 6           # text length (number of tokens)

# ---- Dummy Image ----
# a bright square on dark background, just so it's visually non-random
dummy_image = tf.zeros((128,128,3), dtype=tf.float32)
dummy_image = tf.tensor_scatter_nd_update(
    dummy_image,
    indices=[[32,32],[32,96],[96,32],[96,96]],
    updates=[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], # Fixed: updates must be 3-element vectors
)
dummy_image = tf.expand_dims(dummy_image, 0)   # shape (1,128,128,3)

print("Dummy image shape:", dummy_image.shape)

# ---- Dummy Text for Decoder (Caption) ----
# Pretend the caption is: "<bos> the square is bright <eos>"
# (In reality we use integer tokens 123, 45, 67, ...)
decoder_input_ids = tf.constant([[10, 123, 45, 67, 89, 11]])  # <bos> words
decoder_target_ids = tf.constant([[123, 45, 67, 89, 11, 12]]) # next-token targets

print("Decoder input IDs:", decoder_input_ids)
print("Decoder target IDs:", decoder_target_ids)

# ---- Dummy Text for Encoder (for ITC + ITM) ----
# Pretend an encoding-friendly text like: "a bright square"
encoder_input_ids = tf.constant([[33, 44, 55, 66, 0, 0]])

print("Encoder input IDs:", encoder_input_ids)

# ============================================================
# Forward pass only (no training)
# ============================================================

logits, vis_tokens, txt_embeds = model(dummy_image,
                                       decoder_input_ids,
                                       encoder_input_ids)
pred_token_ids = tf.argmax(logits, axis=-1, output_type=tf.int32)

print("\nFORWARD PASS RESULTS")
print("--------------------------------")
print("Logits shape:", logits.shape)             # (1, T, vocab)
print("Visual tokens shape:", vis_tokens.shape)  # (1, 32, 256)
print("Text embeds shape:", txt_embeds.shape)    # (1, 256)
print("Predicted token IDs:", pred_token_ids.numpy())

# ============================================================
# One training step using all 3 BLIP-2 losses
# ============================================================

loss, l_itg, l_itc, l_itm = train_step(
    dummy_image,
    decoder_input_ids,
    decoder_target_ids,
    encoder_input_ids
)

print("\nTRAIN STEP RESULTS (BLIP-2 LOSSES)")
print("--------------------------------")
print("Total Loss:", float(loss))
print("ITG Loss:", float(l_itg))
print("ITC Loss:", float(l_itc))
print("ITM Loss:", float(l_itm))

Dummy image shape: (1, 128, 128, 3)
Decoder input IDs: tf.Tensor([[ 10 123  45  67  89  11]], shape=(1, 6), dtype=int32)
Decoder target IDs: tf.Tensor([[123  45  67  89  11  12]], shape=(1, 6), dtype=int32)
Encoder input IDs: tf.Tensor([[33 44 55 66  0  0]], shape=(1, 6), dtype=int32)

FORWARD PASS RESULTS
--------------------------------
Logits shape: (1, 6, 1000)
Visual tokens shape: (1, 32, 256)
Text embeds shape: (1, 256)
Predicted token IDs: [[133 244 969 768 366 248]]

TRAIN STEP RESULTS (BLIP-2 LOSSES)
--------------------------------
Total Loss: 7.483669281005859
ITG Loss: 7.12499475479126
ITC Loss: 0.0
ITM Loss: 0.7173490524291992
